## Local Function Call Example

Tool calls (or function calls) enable LLMs to access capabilities and information that was not available when they were trained.  You can easily configure commercial models to talk with tools.  These same capabilities have moved downstream and are also available in small language models if they have been trained to perform them.  Increasingly this a regular part of model training, usually in the mid training phase.  You can think of tools as being like a function call in your code. Hugging Face discusses their mechanism [here](https://huggingface.co/docs/hugs/guides/function-calling).  This notebook demonstrates extending an SLM -- [OLMO 3 7B Instruct](https://huggingface.co/allenai/Olmo-3-7B-Instruct) -- model with a calculator.  The calculator is just a python function that takes two inputs and an operand and returns a result.

## 1. Setup and Install Dependencies

In [1]:
# We need the latest transformers and accelerate to run OLMo 3.
# Installing from source is recommended for the newest models like OLMo 3.
!pip install -q -U git+https://github.com/huggingface/transformers accelerate bitsandbytes

import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


## 2. Define the Calculator Tool

This is the python function that does the actual arithmetic.  Look at Section 4 to see the configuration that happens inside the LLM to get the function executed.

In [2]:

# This tool only allows specific mathematical operations and validates inputs.
# Note: It is HIGHLY unwise to simply execute code from strangers without checking it

def calculator_tool(operation, x, y):
    """
    Performs basic arithmetic operations.
    Args:
        operation (str): One of 'add', 'subtract', 'multiply', 'divide'.
        x (float/int): First number.
        y (float/int): Second number.
    Returns:
        float or str: Result of calculation or error message.
    """
    # 1. Validate allowed operations
    allowed_ops = ['add', 'subtract', 'multiply', 'divide']
    if operation not in allowed_ops:
        return f"Error: Invalid operation '{operation}'. Only {allowed_ops} are allowed."

    # 2. Validate input types (must be numbers)
    # We also attempt to convert strings if the model passes them for large numbers
    try:
        if isinstance(x, str):
            x = float(x.replace(',', ''))
            if x.is_integer(): x = int(x)
        if isinstance(y, str):
            y = float(y.replace(',', ''))
            if y.is_integer(): y = int(y)
    except ValueError:
        return f"Error: Inputs must be valid numbers."

    if not (isinstance(x, (int, float)) and isinstance(y, (int, float))):
        return f"Error: Inputs must be numbers. Received x={type(x)}, y={type(y)}"

    # 3. Perform calculation
    try:
        if operation == 'add':
            return x + y
        elif operation == 'subtract':
            return x - y
        elif operation == 'multiply':
            return x * y
        elif operation == 'divide':
            if y == 0:
                return "Error: Division by zero is not allowed."
            return x / y
    except Exception as e:
        return f"Error during calculation: {str(e)}"

# Wrapper to execute tool from JSON arguments
def execute_tool_call(tool_json):
    try:
        data = json.loads(tool_json)
        if data.get("name") == "calculator":
            # Check if arguments are nested (as requested) or flat (common model behavior)
            if "arguments" in data:
                args = data["arguments"]
            else:
                args = data

            result = calculator_tool(
                args.get("operation"),
                args.get("x"),
                args.get("y")
            )
            return result
        else:
            return "Error: Unknown tool."
    except json.JSONDecodeError:
        return "Error: Invalid JSON format from model."

print("Calculator tool defined successfully.")

Calculator tool defined successfully.


## 3. Load OLMo 3 7B Instruct Model

As part of the training process, this model was trained to  trigger and accept results from function calls.  You can read about that process as well as a discussion of the training data in Section 5 of this [technical report](https://arxiv.org/pdf/2512.13961).

In [3]:
# We use 4-bit quantization (via bitsandbytes) to ensure it fits comfortably in a standard Colab GPU.
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,

)

# Now load the model
model_id = "allenai/Olmo-3-7B-Instruct"

print(f"Loading {model_id}... this may take a minute.")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")


Loading allenai/Olmo-3-7B-Instruct... this may take a minute.


config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.32k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/204 [00:00<?, ?B/s]

Model loaded successfully!


## 4. Define Inference Function with Tool Logic

For demonstration purposes, the function contains code to allow for an enable_tool toggle in order to run the examples below.

### `run_query` orchestrates tool use

We need some way of actually invoking the model with the prompt e.g. calling model.generate().  This function represents a simplified **Agentic Control Loop**. We could also use LangGraph or LangChain to do the same.

Here is the step-by-step breakdown of the orchestration:

1.  **Schema Injection & Interface Definition**:
    *   We define `tool_schema` as a rigid JSON structure. This serves as the "API Signature" for the model.
    *   **Prompt Engineering**: Inside the `system_prompt`, we do not just ask the model to use the tool; we force a specific syntax. By providing **Few-Shot Examples** (e.g., `User: Add... Assistant: {...}`), we condition the model's attention mechanism to attend to patterns where mathematical queries result in JSON outputs.

2.  **Context Formatting (`apply_chat_template`)**:
    *   Raw strings aren't enough. Models are fine-tuned on specific formats (e.g., specific control tokens like `<|user|>` or `[INST]`). The tokenizer maps our list of dictionaries to this exact expected string format so the model recognizes the distinction between System instructions and User queries.

3.  **Deterministic Inference (`do_sample=False`)**:
    *   We use **Greedy Decoding**. When calling a tool, we want the most probable token at every step to ensure valid JSON syntax. High temperature (randomness) increases the risk of the model hallucinating keys or breaking JSON formatting.

4.  **The Execution Logic**:
    *   **Detection**: We check if the generated response starts with `{` and contains the tool name. This is a heuristic heuristic for intent detection.
    *   **Delegation**: If a tool call is detected, the string is parsed into a dictionary and passed to the Python function (`execute_tool_call`). In a full "ReAct" loop, this result would be appended to the chat history and fed back to the LLM to generate a final natural language response.

In [4]:

def run_query(user_query, enable_tool=True):
    # Definition of the tool schema to pass to the model
    tool_schema = {
        "name": "calculator",
        "description": "Perform basic math. Use this for any calculation. Supports large numbers and high-precision floating point values.",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
                "x": {"type": "number"},
                "y": {"type": "number"}
            },
            "required": ["operation", "x", "y"]
        }
    }

    # Construct the System Prompt
    if enable_tool:
        system_prompt = f"""You are a helpful assistant. You have access to the following tool:
{json.dumps(tool_schema)}

To use the tool, you MUST respond with ONLY a JSON object in this format:
{{"name": "calculator", "arguments": {{"operation": "...", "x": ..., "y": ...}}}}

Here are examples of correct behavior:
User: Add 12345.6789 and 98765.4321
Assistant: {{"name": "calculator", "arguments": {{"operation": "add", "x": 12345.6789, "y": 98765.4321}}}}

User: Calculate 9876543.21 multiplied by 0.123
Assistant: {{"name": "calculator", "arguments": {{"operation": "multiply", "x": 9876543.21, "y": 0.123}}}}

IMPORTANT: Copy numbers EXACTLY from the user query into the JSON. Do not round, truncate, or shift decimal points. Check your work.

If the user asks a question requiring math, generate the tool call JSON.
If the user asks a general question, answer normally.
"""
    else:
        system_prompt = "You are a helpful assistant. You do not have access to any external tools. Answer to the best of your ability."

    # Format using the tokenizer's chat template
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]

    # Convert to input tokens
    # We force return_dict=True to get a BatchEncoding, and then unpack it for generate
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False, # Use greedy decoding for deterministic tool calls
        # temperature=0.1 # Removed temperature as it is invalid when do_sample=False
    )

    # Decode output
    # We slice off the input tokens to see only what the model generated
    input_len = inputs.input_ids.shape[1]
    response_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    print(f"\n{'='*20}\nQuery: {user_query}")
    print(f"Tool Enabled: {enable_tool}")
    print(f"Model Raw Response: {response_text}")

    # Check if the model is trying to call the tool (Simple JSON detection)
    if enable_tool and response_text.startswith("{") and "calculator" in response_text:
        print("--- Detected Tool Call ---")
        result = execute_tool_call(response_text)
        print(f"Tool Execution Result: {result}")
        # Ideally, you would feed this result back to the model, but for this demo we stop here.
    else:
        print("--- No Tool Call Detected ---")

## 5. Run Scenario 1 - Tool Calling

In [5]:
# Scenario 1: Tool Calling Capability
# We ask a math question that is hard for a language model to do purely token-by-token without a tool.
prompt_with_tool = "Calculate 9245617.20354 multiplied by 8.23."
#prompt_with_tool = "Calculate 617.203 multiplied by 8.23."
run_query(prompt_with_tool, enable_tool=True)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Query: Calculate 9245617.20354 multiplied by 8.23.
Tool Enabled: True
Model Raw Response: {"name": "calculator", "arguments": {"operation": "multiply", "x": 9245617.20354, "y": 8.23}}
--- Detected Tool Call ---
Tool Execution Result: 76091429.5851342


In [6]:
#let's just confirm
9245617.20354 * 8.23

76091429.5851342

## 6. Run Scenario 2 - General Knowledge

In [8]:
# Scenario 2: No Tool Calling (General Knowledge)
# We disable the tool instructions. The model will try to answer directly (and might hallucinate or get it right depending on difficulty).
prompt_without_tool = "What represents the letter 'A' in the NATO phonetic alphabet?"
run_query(prompt_without_tool, enable_tool=False)


Query: What represents the letter 'A' in the NATO phonetic alphabet?
Tool Enabled: False
Model Raw Response: The letter 'A' in the NATO phonetic alphabet is represented by the word **"Alpha"**.
--- No Tool Call Detected ---


## 7. Run Scenario 3 - LLM without Tool

In [9]:
# Scenario 3: Math without tool (showing the difference)
# This shows what happens if we ask the math question but forbid the tool.
run_query(prompt_with_tool, enable_tool=False)


Query: Calculate 9245617.20354 multiplied by 8.23.
Tool Enabled: False
Model Raw Response: Let's calculate \( 9,245,617.20354 \times 8.23 \) step by step.

### Step 1: Break down the multiplication

We can write:
\[
9,245,617.20354 \times 8.23 = 9,245,617.20354 \times (8 + 0.23)
\]

So, compute each part separately and add them.

---

#### Part 1: \( 9,245,617.20354 \times 8 \)

\[
9,245,617.20354 \times 8 = (9,245,617.20354 \times 10) - (9,245,617.20354 \times 2)
\]

But easier: just multiply directly.

\[
9,245,617.20354 \times 8 = 74,365,337.628272
\]

---

#### Part 2: \( 9,245,617.20354 \times 0.23 \)

First, \( 9,245,617.20354 \times 0.2 = 1,849,123.440708 \)

Then, \( 9,245,617.20354 \
--- No Tool Call Detected ---
